In [1]:
from kinbert_dataset import BertDataset, BertLabeledDataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import math
from functools import reduce

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from tape import ProteinBertForSequenceClassification, TAPETokenizer
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt

from kinbert_mlm import MLM

from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
import seaborn as sn
import pandas as pd

from eval_utils import get_num_correct, createConfusionMatrix

In [2]:
print(torch.cuda.device_count())
print(torch.cuda.is_available())
print(torch.version.cuda)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
torch.cuda.empty_cache() 

1
True
11.6


In [3]:
# for vulcan
MODEL_DIR = "../../model/"
BASE_MODEL = "tapePretrainSep_interaction_base_20e_batchsize_8_lr1e5"
MODEL_TAG = "tapeClassify"
DATA_DIR = "../../data/"
DATA_NAME_SUB = "ptm_unmapped_substrates.pkl"
DATA_NAME_KIN = "human_kinome_seq.pkl"

N_EPOCH = 30
BATCH_SIZE = 16

LR = "1e3"
lr=1e-3


In [4]:
model = ProteinBertForSequenceClassification.from_pretrained("../../model/tapeClassify_interations_e2e_1n1p_classed_random_30e_batchsize_8_lr_1e3_15aa").to(device)
tokenizer = TAPETokenizer(vocab='iupac')

In [5]:
# dat

In [4]:
dat_sub = pd.read_pickle(os.path.join(DATA_DIR,DATA_NAME_SUB))
dat_kin = pd.read_pickle(os.path.join(DATA_DIR,DATA_NAME_KIN))

In [5]:
dat_sub.columns = ['substrate_gene', 'substrate', 'wt_aa', 'mut_ind', 'mut_aa', 'var_type',
       'var_class', 'var_dist', 'p_ind', 'Protein names', 'sub_organism',
       'substrate_seq']

In [6]:
dat_sub.to_pickle(os.path.join(DATA_DIR,DATA_NAME_SUB))

In [8]:
kin_name_ls = []
logit_array_ls = []

for i in range(245,dat_kin.shape[0]):
    s = dat_kin.iloc[[i],:]
    print(s["kinase"].values[0])
    kin_name_ls.append(s["kinase"].values[0])
    dat_sub["kinase"] = s["kinase"].values[0]
    dat_sub["kinase_seq"] = s["kinase_seq"].values[0]
    dat_sub["kinase_coord"] = s["kinase_coord"].values[0][0][0]+"-"+s["kinase_coord"].values[0][0][1]
    dat_sub["kinase_coord"] = dat_sub["kinase_coord"].apply(lambda s: [s.split("-")])
    dat = dat_sub
    dat["label"] = 1
    dataset = BertLabeledDataset(dat,psite_env = 7,check_psite = False)
    dataloader = DataLoader(dataset, 
                          batch_size=BATCH_SIZE, 
                          shuffle=False, 
                          collate_fn=dataset.collate_fn)
    running_vloss = 0.0
    total_vcorrect = 0
    n_vsample = 0
    model.train(False) # Don't need to track gradents for validation
    logits_preds = []
    y_preds = [] # save predction
    y_trues = [] # save ground truth
    class_probs = []
    class_labels = []

    for j, vdata in enumerate(tqdm(dataloader), 0):
        batch_tokens, input_mask, targets = vdata['input_ids'], vdata['input_mask'], vdata['label']
        (vloss, metrics),logits = model(input_ids = batch_tokens.to(device), input_mask=input_mask.to(device), targets=targets.to(device))
        # total_vcorrect += get_num_correct(logits.to(device), targets.to(device))
        n_vsample += batch_tokens.shape[0]
        y_pred = (torch.max(torch.exp(logits), 1)[1]).data.cpu().numpy()
        y_preds.extend(y_pred)  # save prediction
        y_true = targets.data.cpu().numpy()
        y_trues.extend(y_true)
        logits_preds.append(logits.data.cpu().numpy())

    logits_preds = np.concatenate(logits_preds)
    logit_array_ls.append(logits_preds)

Q14680


100%|██████████| 1349/1349 [00:42<00:00, 31.54it/s]


Q12866


100%|██████████| 1349/1349 [00:46<00:00, 28.84it/s]


P08581


...

100%|██████████| 1349/1349 [00:40<00:00, 33.34it/s]


Q9H1R3


100%|██████████| 1349/1349 [00:42<00:00, 31.74it/s]


Q15746


100%|██████████| 1349/1349 [00:42<00:00, 31.75it/s]


In [9]:
len(logit_array_ls)

227

In [10]:
# logit_array_ls[0:244]

In [11]:
# kin_name_ls[0:244]

In [12]:
import pickle
# with open('../../data/ptmvar_logits_1.pkl', 'wb') as f:
#     pickle.dump(logit_array_ls[0:244], f)
    
# with open('../../data/ptmvar_kinasename_1.pkl', 'wb') as f:
#     pickle.dump(kin_name_ls[0:244], f)
    
with open('../../data/ptmvar_logits_2.pkl', 'wb') as f:
    pickle.dump(logit_array_ls, f)
    
with open('../../data/ptmvar_kinasename_2.pkl', 'wb') as f:
    pickle.dump(kin_name_ls, f)

In [13]:
with open('../../data/ptmvar_logits_1.pkl', 'rb') as f:
    logit_1 = pickle.load(f)
    
with open('../../data/ptmvar_logits_2.pkl', 'rb') as f:
    logit_2 = pickle.load(f)

In [14]:
with open('../../data/ptmvar_kinasename_1.pkl', 'rb') as f:
    kin_name_ls_1 = pickle.load(f)
    
with open('../../data/ptmvar_kinasename_2.pkl', 'rb') as f:
    kin_name_ls_2 = pickle.load(f)

In [17]:
logit_all = logit_1 + logit_2

In [19]:
kin_name_ls_all = kin_name_ls_1 + kin_name_ls_2

In [20]:
len(kin_name_ls_all)

471